# Real Estate Housing Price Prediction
## Ridge and Lasso Regression

**Objective:** Build and compare Ridge Regression and Lasso Regression models to predict median house values using the California Housing dataset.

**Environment:** Google Colab / Jupyter Notebook

**Note:** The target `MedHouseVal` is measured in units of $100,000. This is a public educational dataset, not a dataset of current individual property listings.


## 1. Install and import required libraries


In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib seaborn joblib

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Libraries imported successfully.')


## 2. Load the California Housing dataset
The dataset contains housing-related features and a median house value target. The first run may download the dataset from the internet.


In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print('Dataset loaded successfully!')
print('Shape:', df.shape)
display(df.head())
print('\nColumn names:')
print(df.columns.tolist())
print('\nTarget column:', housing.target_names)


## 3. Understand and clean the data
Check data types, missing values, and duplicate rows before modelling.


In [ ]:
print('Dataset information:')
df.info()

print('\nMissing values per column:')
display(df.isnull().sum().to_frame('Missing values'))

print('Duplicate rows:', df.duplicated().sum())

df = df.drop_duplicates().copy()
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna().copy()
print('Shape after cleaning:', df.shape)


## 4. Exploratory Data Analysis (EDA)
Review summary statistics and visualize the target distribution and feature correlations.


In [ ]:
display(df.describe().T)

plt.figure(figsize=(8, 5))
sns.histplot(df['MedHouseVal'], kde=True)
plt.title('Distribution of Median House Values')
plt.xlabel('Median house value (units of $100,000)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()


## 5. Feature engineering
Create a few household-level ratios. Replace invalid values and remove rows where engineered values are not finite.


In [ ]:
df['RoomsPerHousehold'] = df['AveRooms'] / df['AveOccup'].replace(0, np.nan)
df['BedroomsPerRoom'] = df['AveBedrms'] / df['AveRooms'].replace(0, np.nan)
df['PopulationPerHousehold'] = df['Population'] / df['AveOccup'].replace(0, np.nan)

df = df.replace([np.inf, -np.inf], np.nan).dropna().copy()
print('Feature engineering completed.')
print('Updated dataset shape:', df.shape)
display(df.head())


## 6. Split features and target (80/20)
Use 80% of the rows for training and 20% for testing. The target is `MedHouseVal`.


In [ ]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))
print('Number of input features:', X_train.shape[1])


## 7. Train Ridge and Lasso Regression models
StandardScaler is placed inside each pipeline so scaling is fitted only on the training data.


In [ ]:
ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
lasso_model = make_pipeline(StandardScaler(), Lasso(alpha=0.01, max_iter=20000))

models = {
    'Ridge Regression': ridge_model,
    'Lasso Regression': lasso_model
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, 'trained successfully.')


## 8. Evaluate model performance
Compare Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and R-squared (R²). Lower error values and a higher R² generally indicate better performance.


In [ ]:
results = []
predictions_by_model = {}

for name, model in models.items():
    predictions = model.predict(X_test)
    predictions_by_model[name] = predictions
    mse = mean_squared_error(y_test, predictions)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, predictions),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2 Score': r2_score(y_test, predictions)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))


## 9. Visualize model comparison


In [ ]:
ax = results_df.set_index('Model')[['MAE', 'RMSE']].plot(kind='bar', figsize=(9, 5))
ax.set_title('Error Metric Comparison: Ridge vs Lasso')
ax.set_ylabel('Error (target units)')
ax.set_xlabel('Model')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

ax = results_df.set_index('Model')[['R2 Score']].plot(kind='bar', figsize=(7, 4), ylim=(0, 1))
ax.set_title('R² Score Comparison: Ridge vs Lasso')
ax.set_ylabel('R² score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 10. Residual error analysis
A residual is the actual target value minus the predicted value. A residual plot helps reveal patterns in prediction errors.


In [ ]:
best_model_name = results_df.loc[results_df['R2 Score'].idxmax(), 'Model']
best_model = models[best_model_name]
best_predictions = predictions_by_model[best_model_name]
residuals = y_test.to_numpy() - best_predictions

print('Best model by test R²:', best_model_name)

plt.figure(figsize=(8, 5))
plt.scatter(best_predictions, residuals, alpha=0.35)
plt.axhline(y=0, linestyle='--')
plt.title('Residual Plot - ' + best_model_name)
plt.xlabel('Predicted median house value')
plt.ylabel('Residual (actual - predicted)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True)
plt.title('Distribution of Residual Errors')
plt.xlabel('Residual')
plt.tight_layout()
plt.show()


## 11. Actual vs predicted values


In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.35)
low = min(y_test.min(), best_predictions.min())
high = max(y_test.max(), best_predictions.max())
plt.plot([low, high], [low, high], linestyle='--')
plt.title('Actual vs Predicted House Values - ' + best_model_name)
plt.xlabel('Actual values')
plt.ylabel('Predicted values')
plt.tight_layout()
plt.show()


## 12. Save the best model and evaluation results


In [ ]:
joblib.dump(best_model, 'housing_model.pkl')
results_df.to_csv('housing_model_results.csv', index=False)

print('Saved housing_model.pkl')
print('Saved housing_model_results.csv')
display(results_df)


## 13. Conclusion
In this project, the California Housing dataset was explored and prepared for supervised regression. Ridge Regression and Lasso Regression were trained using standardized input features and evaluated on a held-out test set using MAE, MSE, RMSE, and R². Residual plots were used to inspect prediction errors. The model with the higher test R² is selected as the best model for this experiment. Record the actual metric values displayed by your notebook in your internship report.


## 14. Download files in Google Colab (optional)
Run this cell in Colab if you want to download the model and CSV results to your computer.
